# Price timing score

How well has the household (excl. EV charging) placed its consumption on the cheap hours of each day?

Per Danish day:

- **Actual** = Σ hour kWh × hour price (incl. VAT).
- **Even spread** = day kWh × the day's average hourly price – what the same kWh would have cost spread evenly over the day.
- **Cheapest** = day kWh × the day's lowest price – the theoretical best.
- **Gain** = even − actual (kr saved by timing; negative = timing cost money).
- **Capture** = gain ÷ (even − cheapest): the share of the possible saving that was taken.
- **Cheap 6 h** = share of kWh in the day's 6 cheapest hours (even spread = 25 %).
- **Evening 17–21** = share of kWh from 17:00 to 20:59 (even spread = 17 %).

Only days with at least 23 priced hours are counted. The data comes from the view `household_hourly_price` (see `household_hourly_price.sql`).

The password is read from the `MySqlConnection__Password` environment variable (the same one MultiApiFetcher and PowerConsumption use); if it isn't set, you are asked for it.

In [ ]:
import os
from getpass import getpass

import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from sqlalchemy import create_engine, URL

DB_HOST = "127.0.0.1"
DB_PORT = 3306
DB_NAME = "electricity_db"
DB_USER = "perh"

MIN_HOURS = 23          # a day needs this many priced hours to count
CHEAP_HOURS = 6         # "cheap" = the day's N cheapest hours
EVENING = range(17, 21) # 17:00-20:59 local time

password = os.environ.get("MySqlConnection__Password") or getpass(f"Password for {DB_USER}@{DB_HOST}: ")
engine = create_engine(URL.create("mysql+pymysql", username=DB_USER, password=password,
                                  host=DB_HOST, port=DB_PORT, database=DB_NAME))
pd.options.display.float_format = "{:,.2f}".format

In [ ]:
raw = pd.read_sql(
    "SELECT hour_utc, household_kwh, ev_kwh, price_kr_incl_vat FROM household_hourly_price ORDER BY hour_utc",
    engine)

df = raw.rename(columns={"household_kwh": "kwh", "price_kr_incl_vat": "price"})
df["kwh"] = df["kwh"].astype(float)
df["price"] = df["price"].astype(float)

# UTC in the database -> Danish time (handles the 23- and 25-hour days).
local = pd.to_datetime(df["hour_utc"]).dt.tz_localize("UTC").dt.tz_convert("Europe/Copenhagen")
df["day"] = local.dt.date
df["hour"] = local.dt.hour
df["cost"] = df["kwh"] * df["price"]

# Rank the hours of each day by price; ties go to the earliest hour.
df["price_rank"] = df.groupby("day")["price"].rank(method="first")
df["cheap_kwh"] = df["kwh"].where(df["price_rank"] <= CHEAP_HOURS, 0)
df["evening_kwh"] = df["kwh"].where(df["hour"].isin(EVENING), 0)

print(f"{len(df):,} hours from {df['day'].min()} to {df['day'].max()}, "
      f"household {df['kwh'].sum():,.1f} kWh, EV {df['ev_kwh'].astype(float).sum():,.1f} kWh, "
      f"cost {df['cost'].sum():,.2f} kr")

In [ ]:
daily = df.groupby("day").agg(
    hours=("price", "size"),
    kwh=("kwh", "sum"),
    actual=("cost", "sum"),
    avg_price=("price", "mean"),
    min_price=("price", "min"),
    cheap_kwh=("cheap_kwh", "sum"),
    evening_kwh=("evening_kwh", "sum"),
)
skipped = (daily["hours"] < MIN_HOURS).sum()
daily = daily[daily["hours"] >= MIN_HOURS].copy()
daily["even"] = daily["kwh"] * daily["avg_price"]
daily["cheapest"] = daily["kwh"] * daily["min_price"]
daily.index = pd.to_datetime(daily.index)
print(f"{len(daily)} days counted, {skipped} skipped for missing hours")


def score(groups):
    """Sum days into periods and work out the timing measures."""
    s = groups[["kwh", "actual", "even", "cheapest", "cheap_kwh", "evening_kwh"]].sum()
    s["days"] = groups.size()
    s["paid_kr_kwh"] = s["actual"] / s["kwh"]
    s["even_kr_kwh"] = s["even"] / s["kwh"]
    s["gain_kr"] = s["even"] - s["actual"]
    s["gain_pct"] = 100 * s["gain_kr"] / s["even"]
    s["capture_pct"] = 100 * s["gain_kr"] / (s["even"] - s["cheapest"])
    s["cheap6_pct"] = 100 * s["cheap_kwh"] / s["kwh"]
    s["evening_pct"] = 100 * s["evening_kwh"] / s["kwh"]
    return s[["days", "kwh", "actual", "paid_kr_kwh", "even_kr_kwh", "gain_kr", "gain_pct",
              "capture_pct", "cheap6_pct", "evening_pct"]]

## Per year

In [ ]:
years = score(daily.groupby(daily.index.year))
years.index.name = "year"
years

## Per month

In [ ]:
months = score(daily.groupby(daily.index.to_period("M")))
months.index.name = "month"
months

In [ ]:
labels = months.index.astype(str)
fig = make_subplots(rows=2, cols=1, shared_xaxes=True, vertical_spacing=0.08,
                    subplot_titles=("Gain from timing (kr) – above zero is good",
                                    "Share of kWh in the day's 6 cheapest hours (%)"))
fig.add_bar(x=labels, y=months["gain_kr"], row=1, col=1, name="Gain kr",
            marker_color=["#2a9d8f" if g >= 0 else "#e76f51" for g in months["gain_kr"]],
            hovertemplate="%{x}: %{y:.0f} kr<extra></extra>")
fig.add_scatter(x=labels, y=months["cheap6_pct"], row=2, col=1, name="Cheap 6 h %", mode="lines+markers",
                line_color="#264653", hovertemplate="%{x}: %{y:.1f} %<extra></extra>")
fig.add_hline(y=25, row=2, col=1, line_dash="dot", line_color="grey",
              annotation_text="even spread 25 %", annotation_position="bottom right")
fig.update_layout(height=600, showlegend=False, template="plotly_white", margin=dict(t=60, b=40))
fig.show()

## When in the day (Danish time)

Each hour's share of the household's kWh against the average price in that hour. Pick a year in `PROFILE_YEAR` (or `None` for all years).

In [ ]:
PROFILE_YEAR = None  # e.g. 2026

counted = df[pd.to_datetime(df["day"]).isin(daily.index)]
if PROFILE_YEAR:
    counted = counted[pd.to_datetime(counted["day"]).dt.year == PROFILE_YEAR]
profile = counted.groupby("hour").agg(kwh=("kwh", "sum"), price=("price", "mean"))
profile["share_pct"] = 100 * profile["kwh"] / profile["kwh"].sum()

fig = make_subplots(specs=[[{"secondary_y": True}]])
fig.add_bar(x=profile.index, y=profile["share_pct"], name="Share of kWh %", marker_color="#8ab17d",
            hovertemplate="%{x}:00 – %{y:.1f} %<extra></extra>")
fig.add_scatter(x=profile.index, y=profile["price"], name="Avg price kr/kWh", mode="lines+markers",
                line_color="#e76f51", secondary_y=True, hovertemplate="%{x}:00 – %{y:.2f} kr<extra></extra>")
fig.add_hline(y=100 / 24, line_dash="dot", line_color="grey", annotation_text="even 4.2 %")
fig.update_xaxes(title="Hour", dtick=1)
fig.update_yaxes(title_text="Share of kWh (%)", secondary_y=False)
fig.update_yaxes(title_text="kr/kWh incl. VAT", secondary_y=True)
fig.update_layout(title=f"Hour profile – {PROFILE_YEAR or 'all years'}", height=450, template="plotly_white")
fig.show()

## Make a page

To save the results as one HTML file without the code (run from this folder):

```powershell
.venv\Scripts\jupyter nbconvert --to html --no-input --execute price_timing.ipynb
```

`--execute` needs `MySqlConnection__Password` set, because it can't ask for the password.